# petekIO — working with well data

A well carries a **trajectory** (MD ↔ position), **logs** (MD-indexed curves), and
**tops** (formation picks that define intervals). This notebook loads a well from a
folder of LAS + tops CSV, positions MDs in space, reads logs, and uses the fluent
`well.<top>.<log>` ergonomic to get interval statistics — then broadcasts across a
project of wells.

Install: `pip install petekio` (or `maturin develop`). Run from `examples/`.

In [ ]:
import petekio

geo = petekio.GeoData(unit="ft")
# A well folder ingests every *.las (logs) and *.csv (tops); a vertical
# trajectory spanning the logged MD is built when no survey is supplied.
geo.load_well("15/9-A1", head=(1200.0, 1500.0), kb=0.0, files="data/wells/15_9-A1")
w = geo.well("15/9-A1")
print(w.id, "head=", w.head, "kb=", w.kb)

## Positioning along the trajectory

`xyz(md)` gives the world position `(x, y, z)` at a measured depth — `z` is subsea
TVD (positive down). `tvd(md)` returns just the TVD; `md_at_tvd(tvd)` inverts it.

In [ ]:
print("position at MD 2420:", w.xyz(2420.0))
print("TVD at MD 2420:", w.tvd(2420.0))

## Logs

`log(mnemonic)` returns a `LogView` over the whole curve (case-insensitive on the
**raw** LAS mnemonic). Values are as-stored — e.g. `SUWI` here is in percent;
unit/mnemonic normalisation happens in the data-layer assembly, not on raw reads.

In [ ]:
phi = w.log("PHI")
print("samples:", len(phi))
print("first MDs:   ", phi.md()[:5])
print("first values:", phi.values()[:5])
print("mean porosity:", phi.stats().mean)
print("interpolated at MD 2415:", phi.at_md(2415.0))

## Tops, intervals, and the fluent `well.<top>.<log>`

A top resolves an interval `[top_md, base_md)` (base = the next top, or TD for the
deepest). `well.top("Brent")` returns the `Interval`; `well.brent.ntg` resolves the
interval **and** clips the `NTG` log to it, returning `Stats` in one step.

In [ ]:
brent = w.top("Brent")
print(f"Brent: {brent.top_md} -> {brent.base_md}  ({brent.thickness_md()} thick)")

# Fluent: interval -> log -> Stats, in one expression.
print("Brent mean NTG:", w.brent.ntg.mean)
print("Brent mean PHI:", w.brent.phi.mean)
print("Brent P50 SUWI (percent):", w.brent.suwi.p50)

## Broadcasting across a project

`GeoData.wells` is a broadcastable view: iterate it, `filter(predicate)` it, or
narrow to wells that have a named top with `tops(name)`. Operations apply across
the collection — no per-well loops in your own code beyond consuming the results.

In [ ]:
wells = geo.wells
print("wells in project:", len(wells))

# Per-well Brent NTG across every well that has a Brent top.
for well in wells.tops("Brent").iter():
    print(f"  {well.id}: Brent NTG mean = {well.brent.ntg.mean:.3f}")

# filter takes a predicate over wells (e.g. by id).
subset = wells.filter(lambda well: well.id.startswith("15/9"))
print("matched by predicate:", len(subset))